# 02 — Prepare a coherent dataset with FixDataset

**You will:** create a `prepared` branch with explicit hygiene settings and comment
exact repeated observations. You will inspect what PLEB changed before using the result.
Flags, duplicates, jumps and variant definitions are dataset policy; they are not fit diagnostics.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
common = json.loads((CONFIGS/'common.json').read_text())
classification = CONFIGS/'backend_classes.toml'
lab.write_config(classification, {'classifications':{'legacy':['A'],'new':['B']}})
variants = CONFIGS/'variants.toml'
lab.write_config(variants, {'variants':{
    'legacy':dict(include_classes=['legacy'],exclude_classes=['new'],mixed_policy='error'),
    'new':dict(include_classes=['new'],exclude_classes=['legacy'],mixed_policy='error'),
    'combined':dict(include_classes=['legacy','new'],exclude_classes=[],mixed_policy='any')}})
prepared = dict(common, branches=['prepared'], reference_branch='prepared',
    outdir_name='02_prepare', run_fix_dataset=True, fix_apply=True,
    fix_base_branch='raw_ingest', fix_branch_name='prepared',
    fix_commit_message='Prepare systems and comment exact duplicate occurrences',
    fix_dedupe_toas_within_tim=True, fix_dedupe_action='comment',
    fix_remove_overlaps_exact=False, fix_check_duplicate_backend_tims=False,
    fix_insert_missing_jumps=False, fix_prune_stale_jumps=False, fix_remove_patterns=[],
    fix_generate_alltim_variants=True, fix_backend_classifications_path=str(classification),
    fix_alltim_variants_path=str(variants), fix_jump_reference_variants=False,
    fix_system_flag_update_table=False, fix_standardize_par_values=False)
path = lab.write_config(CONFIGS/'02_prepare.toml',prepared)

`fix_apply=true` creates a branch and commits changes; without it the stage is a report.
`fix_dedupe_action="comment"` retains the original row and records why it is inactive.
This exercise uses exact within-file duplicates. Real overlap rules need an explicit preference
for which observation survives; repeated MJDs across frequencies/backends are not automatically duplicates.

The raw PAR already contains the correct `JUMP -sys B` and uses A as reference.
We preserve that choice. Blindly fitting a free jump for every backend can be degenerate
with phase. Variant INCLUDE files select observations; they do not certify a separately
fitted timing model for each subset. We fit and publish the full canonical pair in this course.

In [ ]:
lab.command(CLI+['--config',path], log=ROOT/'logs/02_prepare.log')
lab.command(['git','-C',REPO,'diff','--stat','raw_ingest','prepared'])
prepared_data = lab.snapshot(REPO,'prepared',ROOT/'exports/02_prepared')
before = lab.inventory(ROOT/'exports/01_ingested/data')
after = lab.inventory(prepared_data)
display(after)
assert after.physical.sum()==before.physical.sum()==580
assert after.active.sum()==576 and after.commented.sum()==4
assert after.extra_active_duplicates.sum()==0
print((prepared_data/PULSARS[0]/'tims/A.tim').read_text().splitlines()[-1])

In [ ]:
for wrapper in sorted((prepared_data/PULSARS[0]).glob('*_all.tim')):
    print(wrapper.name, '\n'+wrapper.read_text())
lab.command(['git','-C',REPO,'diff','raw_ingest','prepared','--',f'data/{PULSARS[0]}/{PULSARS[0]}.par'])

**Your task:** find the exact comment prefix and survivor in a backend TIM. Verify that
a physical row count would miss the change in active selection. Explain which extra
checks would be needed before claiming the same duplicate policy works on real data.

## Revisions and dual-band measurements are different from duplicates

A shared archive name alone is not a duplicate key. Inspect this small synthetic
provider example before choosing an overlap policy. A revised extraction replaces
an older measurement only under an explicit input-version preference. Dual-band TOAs
may be two intended measurements replacing one full-band TOA. Match epoch, frequency,
channelisation, uncertainty, flags and provider version; preserve evidence of the choice.
Keeping the first file encountered can retain an obsolete extraction.

In [ ]:
examples = pd.DataFrame([
    ('archive001','old_full',1400.,'58730.0000000000',1.0),
    ('archive001','revised_full',1400.,'58730.0000000002',0.8),
    ('archive001','dual_low',1330.,'58730.0000000001',1.1),
    ('archive001','dual_high',1450.,'58730.0000000003',1.2),
],columns=['archive','input_version','frequency_MHz','mjd_text','error_us'])
display(examples)
assert examples.archive.nunique()==1 and examples.frequency_MHz.nunique()==3
# This table is a classification exercise, not an extra cleaning script.
# The actual exact duplicates above were commented by PLEB FixDataset.

**Your task:** state which rows survive if the provider approves the dual-band version.
Which backend labels and PAR jumps would need review? Record a version preference before
running overlap removal, and check the resulting INCLUDE selection and counts afterward.